
# MEHP Force Relaxation

This example demonstrates how to use the unified
:class:`~pylimer_tools_cpp.MEHPSimulator` with interchangeable constitutive
:class:`~pylimer_tools_cpp.MEHPForceEvaluator` models and
:class:`~pylimer_tools_cpp.MEHPSolver` strategies (see :cite:t:`gusev_numerical_2019`)
without slip-links or entanglements.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from pylimer_tools.io.bead_spring_parameter_provider import (
    ParameterType,
    get_parameters_for_polymer,
)
from pylimer_tools.io.read_lammps_output_file import read_data_file
from pylimer_tools_cpp import (
    EntanglementMode,
    HookeanForceEvaluator,
    MEHPForceEvaluator,
    MEHPNetwork,
    MEHPNonlinearSolverConfig,
    MEHPSimulator,
    NLoptAlgorithm,
    NonGaussianSpringForceEvaluator,
    NonlinearOptimizationSolver,
    Universe,
)


# Example: Stateless Custom Force Evaluator (see also tests)
# This particular example is only illustrative, as the harmonic spring law is
# already implemented faster in C++ by HookeanForceEvaluator.
class CustomForceEvaluator(MEHPForceEvaluator):
    """A stateless custom force evaluator implementing a simple harmonic spring."""

    def __init__(self, kappa: float = 1.0) -> None:
        super().__init__()
        self.kappa = float(kappa)

    def get_kappa(self) -> float:
        return self.kappa

    def evaluate_spring_energy(
        self, spring_vec: np.ndarray, contour_length: float
    ) -> float:
        vec = np.asarray(spring_vec, dtype=np.float64)
        return float(0.5 * self.kappa * np.dot(vec, vec) / contour_length)

    def evaluate_spring_force(
        self, spring_vec: np.ndarray, contour_length: float
    ) -> np.ndarray:
        vec = np.asarray(spring_vec, dtype=np.float64)
        return (self.kappa / contour_length) * vec


# Load your network (replace with your file)
def _find_fixtures_dir() -> Path:
    for cur in (
        [Path(__file__).resolve().parent] if "__file__" in globals() else []
    ) + [Path.cwd()]:
        for p in [cur] + list(cur.parents):
            candidate = p / "tests" / "pylimer_tools" / "fixtures"
            if candidate.is_dir():
                return candidate
    return Path("tests/pylimer_tools/fixtures").resolve()


_fixtures_dir = _find_fixtures_dir()
universe = read_data_file(
    str(_fixtures_dir / "structure/network_100_a_46.structure.out")
)
assert isinstance(universe, Universe)

# Prepare parameters for conversion factors
params = get_parameters_for_polymer("PDMS", parameter_type=ParameterType.GAUSSIAN)
r02_slope = params.get("R02")
r02_slope_magnitude = r02_slope.to(params.get("distance_units") ** 2).magnitude
kbt = params.get("T") * params.get("kb")
gamma_conversion_factor = (
    (kbt / ((params.get("distance_units")) ** 3)).to("MPa").magnitude
)

base_net = MEHPNetwork.from_universe(universe, entanglement_mode=EntanglementMode.NONE)

# 1. MEHPSimulator with Hookean (linear) force evaluator and default linear solver
sim_linear = MEHPSimulator(base_net, HookeanForceEvaluator())
sim_linear.run()
shear_modulus_linear = (
    gamma_conversion_factor
    * np.sum(sim_linear.get_gamma_factors(r02_slope_magnitude))
    / universe.get_volume()
)
print(f"Phantom Shear Modulus [MPa] (Hookean): {shear_modulus_linear:.4f}")

# 2. MEHPSimulator with Langevin (non-Gaussian) force evaluator and nonlinear solver
nonlinear_cfg = MEHPNonlinearSolverConfig()
nonlinear_cfg.algorithm = NLoptAlgorithm.LD_MMA
sim_langevin = MEHPSimulator(base_net, NonGaussianSpringForceEvaluator())
sim_langevin.run(nonlinear_cfg)
shear_modulus_langevin = (
    gamma_conversion_factor
    * np.sum(sim_langevin.get_gamma_factors(r02_slope_magnitude))
    / universe.get_volume()
)
print(f"Phantom Shear Modulus [MPa] (Langevin): {shear_modulus_langevin:.4f}")

# 3. MEHPSimulator with Custom Python Force Evaluator and explicit solver strategy
lbfgs_cfg = MEHPNonlinearSolverConfig()
lbfgs_cfg.algorithm = NLoptAlgorithm.LD_LBFGS
custom_evaluator = CustomForceEvaluator(kappa=1.0)
sim_custom = MEHPSimulator(base_net, custom_evaluator)
sim_custom.run(NonlinearOptimizationSolver(lbfgs_cfg))
shear_modulus_custom = (
    gamma_conversion_factor
    * np.sum(sim_custom.get_gamma_factors(r02_slope_magnitude))
    / universe.get_volume()
)
print(f"Phantom Shear Modulus [MPa] (Custom Evaluator): {shear_modulus_custom:.4f}")

## Comparison of Force Evaluators
We visualize the predicted equilibrium phantom shear modulus for each
force evaluator.



In [ ]:
labels = ["Hookean\n(Linear)", "Non-Gaussian\n(Langevin)", "Custom\n(Harmonic)"]
moduli = [shear_modulus_linear, shear_modulus_langevin, shear_modulus_custom]
colors = ["#4C72B0", "#55A868", "#DD8452"]

plt.figure(figsize=(7, 5))
bars = plt.bar(
    labels, moduli, color=colors, edgecolor="black", linewidth=0.8, width=0.5
)

for bar in bars:
    yval = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2.0,
        yval + 0.0001,
        f"{yval:.4f} MPa",
        ha="center",
        va="bottom",
        fontsize=10,
    )

plt.ylabel("Phantom Shear Modulus [MPa]", fontsize=11)
plt.title("Comparison of Force Evaluators in MEHPSimulator", fontsize=12)
plt.grid(True, axis="y", linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()